In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os, yaml, torch
from pathlib import Path

print("✅ Environment Ready")
print(f"CUDA Available: {torch.cuda.is_available()}")

## 1. Install Ultralytics

In [ ]:
!pip install -q ultralytics
from ultralytics import YOLO
print("✅ Ultralytics Installed")

## 2. Dynamic Path & YAML Setup
Kaggle datasets are read-only. We must find the absolute path and create a new `data.yaml` in `/kaggle/working`.

In [ ]:
DATASET_NAME = "yolo_dataset_hand_augmented"
SEARCH_ROOT = "/kaggle/input"

def find_dataset(root, target):
    for dirpath, dirnames, filenames in os.walk(root):
        if target in dirnames:
            return os.path.join(dirpath, target)
    return None

DATASET_PATH = find_dataset(SEARCH_ROOT, DATASET_NAME)

if DATASET_PATH:
    print(f"🚀 Dataset found at: {DATASET_PATH}")
    
    # Define absolute paths for YOLO
    train_path = os.path.join(DATASET_PATH, "train/images")
    val_path   = os.path.join(DATASET_PATH, "val/images")
    test_path  = os.path.join(DATASET_PATH, "test/images")
    
    # Create dynamic data.yaml
    data_config = {
        'path': DATASET_PATH,
        'train': 'train/images',
        'val': 'val/images',
        'test': 'test/images',
        'nc': 1,
        'names': ['fracture']
    }
    
    with open('/kaggle/working/data_kaggle.yaml', 'w') as f:
        yaml.dump(data_config, f, default_flow_style=False)
    
    print("📝 Created /kaggle/working/data_kaggle.yaml")
else:
    print("❌ Dataset NOT found. Please check your upload name.")

## 3. Training YOLOv8m (Medium)
Using the Medium model to capture more detail than the Small version, with an image size of 1024px for fine X-ray textures.

In [ ]:
model = YOLO('yolov8m.pt') # Load Medium model

results = model.train(
    data='/kaggle/working/data_kaggle.yaml',
    epochs=100,
    imgsz=1024,
    batch=16,
    patience=20,
    device=0,      # Use first GPU
    project='/kaggle/working/runs',
    name='fracture_detection_yolov8m',
    exist_ok=True,
    # Medical specific augmentations
    degrees=10.0,
    fliplr=0.5,
    mosaic=0.5,
    mixup=0.1
)

## 4. Evaluate & Visualize

In [ ]:
import matplotlib.pyplot as plt
import cv2

run_path = "/kaggle/working/runs/fracture_detection_yolov8m"

# Show metrics
res_img = os.path.join(run_path, "results.png")
if os.path.exists(res_img):
    img = cv2.imread(res_img)
    plt.figure(figsize=(15, 10))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.show()

# Show Confusion Matrix
conf_img = os.path.join(run_path, "confusion_matrix.png")
if os.path.exists(conf_img):
    img = cv2.imread(conf_img)
    plt.figure(figsize=(10, 10))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.show()